# 01. Генерация ответов и перплексия

Этап 0. Для каждой конфигурации: ответы на оценочные промпты (`generations.jsonl`) и перплексия на WikiText-2.

Запускать в Colab на T4. Готовые шаги пропускаются, так что после обрыва сессии ноутбук можно просто запустить заново.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys
REPO = "https://github.com/Kaban17/qsi.git"
if not os.path.exists("/content/qsi"):
    !git clone -q {REPO} /content/qsi
else:
    !git -C /content/qsi pull -q
%cd /content/qsi
!pip install -q -e . bitsandbytes accelerate
sys.path.insert(0, "/content/qsi")

In [ ]:
# Нужен секрет Colab с именем HF_TOKEN и принятая лицензия Gemma на Hugging Face.
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
from pathlib import Path
from transformers import AutoTokenizer
from qsi.data import load_splits, load_wikitext_chunks
from qsi.models import CONFIG_NAMES, MODEL_ID
from qsi.pipeline import truncate_splits
from qsi.results import read_metrics

# FULL = False: дымовой прогон на 5 промптах и двух конфигурациях (несколько минут).
# FULL = True: полный прогон всех 8 конфигураций. Значение должно совпадать во всех трёх ноутбуках.
FULL = False

LIMIT = None if FULL else 5
CONFIGS = CONFIG_NAMES if FULL else ["fp16", "rtn-4"]
RESULTS = Path("/content/drive/MyDrive/qsi") / ("results" if FULL else "results_smoke")

splits = truncate_splits(load_splits(), LIMIT)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
wikitext = load_wikitext_chunks(tokenizer, n_chunks=4 if LIMIT else 40)
{k: len(v) for k, v in splits.items()}, wikitext.shape

In [ ]:
import time
from qsi.pipeline import run_stage0

for name in CONFIGS:
    start = time.time()
    run_stage0(name, RESULTS, splits, wikitext)
    print(f"{name}: {time.time() - start:.0f} s")

In [ ]:
import pandas as pd
pd.DataFrame({name: {"perplexity": read_metrics(RESULTS / name).get("perplexity")} for name in CONFIGS}).T

In [ ]:
# Пара ответов глазами (первые два вредных и два безобидных промпта первой конфигурации).
import json
rows = [json.loads(l) for l in (RESULTS / CONFIGS[-1] / "generations.jsonl").read_text().splitlines()]
for r in rows[:2] + rows[-2:]:
    print(r["split"], "|", r["prompt"][:70], "\n  ->", r["response"][:200].replace("\n", " "), "\n")

После полного прогона скачайте папку `qsi/results` с Google Drive в корень репозитория и локально запустите судью:

```
uv run python -m qsi.judge results/
```